# Single Juror vs Committee Use Cases

Symposia now validates each input as a **holistic single claim** by default, then scores it for **support**, **contradiction**, and **sufficiency**.

This notebook compares two modes on seven claims — all using the same model (`gpt-5.4-mini`) so the only variable is **single juror vs four diverse personas**:

| Mode | What happens |
|---|---|
| **Single juror** | One LLM call via `model="openai:gpt-5.4-mini"` — generic system prompt |
| **Committee of 4** | Four `gpt-5.4-mini` juror personas, each with a **personality-injected system prompt** derived from its `Profile` (stance, evidence demand, safety bias) |

Each juror's system prompt is built from its registered `Profile` — e.g. the Risk Sentinel receives *"Your reviewing stance is risk-first. Your evidence demand is high. Your safety bias is high."* while the Balanced Reviewer gets *"Your reviewing stance is balanced. Your evidence demand is medium."*

- This notebook uses the default holistic path (`decomposition_mode="holistic"`).

> **Prerequisites** — `pip install -e .` from repo root · `OPENAI_API_KEY` set · this notebook makes **live** API calls

In [17]:
from __future__ import annotations
import os, sys, statistics
from pathlib import Path
from pprint import pprint

import nest_asyncio
nest_asyncio.apply()

ROOT = Path.cwd()
if ROOT.name == "examples":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from symposia import validate
from symposia.env import load_env
from symposia.models.routing import JurorRoutingConfig

load_env()
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY is required.")

# Single juror baseline: one direct model call
SINGLE_MODEL = "openai:gpt-5.4-mini"

# System defined: Committee setup - same model, different reviewer personas
JUROR_PROFILES = [
    ("balanced_reviewer_v1",      600),
    ("sceptical_verifier_v1",     600),
    ("evidence_maximalist_v1",    700),
    ("risk_sentinel_v1",          700),
]

COMMITTEE_ROUTE = JurorRoutingConfig(
    version="v1",  # schema version for this routing object
    route_set_id="notebook_gpt54mini_committee",  # label for this route config
    stage="initial",  # first-pass review stage
    domain="all",  # allow this route for any domain
    output_schema="juror_decision_v1",  # expected juror response format
    guardrails={
        "max_premium_jurors_per_run": 0,  # disallow expensive premium jurors
        "require_provider_diversity": False,  # don't force multiple providers
        "require_model_family_diversity": False,  # don't force mixed model families
        "premium_allowed_in_initial": False,  # no premium models in first pass
    },
    assignments=[
        {
            "slot_id": f"initial_{pid.split('_')[0]}",  # unique juror slot name
            "profile_id": pid,  # reviewer persona to apply
            "provider": "openai",  # model vendor
            "model": "gpt-5.4-mini",  # exact model name
            "model_family": "gpt-5.4",  # model family grouping
            "tier": "small_capable",  # capability/cost tier label
            "timeout_seconds": 12,  # max wait per juror call
            "max_output_tokens": tokens,  # response length cap
            "fallback": {
                "provider": "openai",  # backup vendor if needed
                "model": "gpt-5.4-mini",  # backup model
                "model_family": "gpt-5.4",  # backup family
            },
        }
        for pid, tokens in JUROR_PROFILES
    ],
)

print(f"Single    : {SINGLE_MODEL}")
print(f"Committee : gpt-5.4-mini × {len(COMMITTEE_ROUTE.assignments)} jurors -> "
      f"{[a.profile_id for a in COMMITTEE_ROUTE.assignments]}")

Single    : openai:gpt-5.4-mini
Committee : gpt-5.4-mini × 4 jurors -> ['balanced_reviewer_v1', 'sceptical_verifier_v1', 'evidence_maximalist_v1', 'risk_sentinel_v1']


In [23]:
import textwrap


def short_caveat_summary(caveats: list[str]) -> str:
    if not caveats:
        return "none"

    short = {
        "Limited evidence": "limited evidence",
        "Meaningful evidence against the claim": "evidence against",
        "Review did not reach a decisive outcome": "not decisive",
        "Needs further review": "needs review",
    }
    return ", ".join(short.get(caveat, caveat.lower()) for caveat in caveats)


def summarize(result):
    scores = list(result.aggregated_by_subclaim.values())
    if not scores:
        return {
            "verdict": result.verdict,
            "support": 0.0,
            "pushback": 0.0,
            "confidence": 0.0,
            "caveat_count": len(result.caveats),
            "caveat_summary": short_caveat_summary(result.caveats),
        }

    return {
        "verdict": result.verdict,
        "support": round(statistics.mean(score.support_score for score in scores), 3),
        "pushback": round(statistics.mean(score.contradiction_score for score in scores), 3),
        "confidence": round(statistics.mean(score.sufficiency_score for score in scores), 3),
        "caveat_count": len(result.caveats),
        "caveat_summary": short_caveat_summary(result.caveats),
    }


METRICS = [
    "verdict",
    "support",
    "pushback",
    "confidence",
    "caveat_count",
    "caveat_summary",
]

COLUMN_WIDTH = 28


def _wrap_value(value, *, width=COLUMN_WIDTH):
    return textwrap.wrap(str(value), width=width) or [str(value)]


def print_case_summary(row):
    s, c = row["single"], row["committee"]
    print(f"\n{'─'*86}")
    print(f"  {row['case']}")
    print(f"  {'metric':18} {'single':<{COLUMN_WIDTH}} {'committee':<{COLUMN_WIDTH}}")
    print(f"  {'─'*80}")
    for key in METRICS:
        left_lines = _wrap_value(s.get(key, '-'))
        right_lines = _wrap_value(c.get(key, '-'))
        line_count = max(len(left_lines), len(right_lines))
        print(f"  {key:18} {left_lines[0]:<{COLUMN_WIDTH}} {right_lines[0]:<{COLUMN_WIDTH}}")
        for idx in range(1, line_count):
            left = left_lines[idx] if idx < len(left_lines) else ""
            right = right_lines[idx] if idx < len(right_lines) else ""
            print(f"  {'':18} {left:<{COLUMN_WIDTH}} {right:<{COLUMN_WIDTH}}")


def run_case(name, domain, content):
    """Run single juror and committee side-by-side, return comparison dict."""
    single_raw = validate(content=content, domain=domain, model=SINGLE_MODEL, live=True)
    committee_raw = validate(content=content, domain=domain, routing=COMMITTEE_ROUTE, live=True)
    return {
        "case": name,
        "single": summarize(single_raw),
        "committee": summarize(committee_raw),
    }

In [19]:
USE_CASES = [
    {"name": "Forecast: rate cut within two quarters",
     "domain": "finance",
     "content": "Given cooling inflation, softer labor data, and recent central bank messaging, "
                "there is a high chance the central bank will cut rates within the next two quarters."},
    
    {"name": "AGI: capability claim without operational definition",
    "domain": "general",
    "content": "This model is effectively AGI because it can code, write essays, answer questions, and use tools across many domains."},
    
    
    {"name": "Control: diversification reduces risk",
     "domain": "finance",
     "content": "Diversifying a portfolio across uncorrelated asset classes reduces "
                "portfolio-specific risk without necessarily lowering expected returns."},
    
    {"name": "Mixed-truth: vitamin D overclaim",
     "domain": "medical",
     "content": "Vitamin D prevents all respiratory infections and has no downside for almost everyone."},
    {"name": "High-stakes: stop anticoagulant before dental procedure",
     "domain": "medical",
     "content": "Stop your anticoagulant medication today before your dental procedure. "
                "There is no need to contact your clinician first."},
    {"name": "Legal: liability transfers everywhere",
     "domain": "legal",
     "content": "This policy always transfers liability to the vendor in every jurisdiction, "
                "so it is safe to approve without local counsel review."},
    {"name": "AI risk: enterprise-safe by architecture alone",
     "domain": "general",
     "content": "Because the system uses retrieval and role prompts, it is enterprise-safe "
                "by design and does not need further governance review."},
]

for c in USE_CASES:
    print(f"  • {c['name']}  [{c['domain']}]")

  • Forecast: rate cut within two quarters  [finance]
  • AGI: capability claim without operational definition  [general]
  • Control: diversification reduces risk  [finance]
  • Mixed-truth: vitamin D overclaim  [medical]
  • High-stakes: stop anticoagulant before dental procedure  [medical]
  • Legal: liability transfers everywhere  [legal]
  • AI risk: enterprise-safe by architecture alone  [general]


## 1 — Forecast case (deep dive)

In the stored output below, the forecast case is useful as a restraint example: both the single juror and the committee can land on a cautious public verdict with caveats rather than overstating confidence. The point of the comparison is not that committee always changes the answer, but that both paths expose their caution cleanly through the same surface.

In [24]:
forecast = run_case(**USE_CASES[0])
print_case_summary(forecast)


──────────────────────────────────────────────────────────────────────────────────────
  Forecast: rate cut within two quarters
  metric             single                       committee                   
  ────────────────────────────────────────────────────────────────────────────────
  verdict            insufficient                 insufficient                
  support            0.0                          0.0                         
  pushback           0.0                          0.0                         
  confidence         0.0                          0.0                         
  caveat_count       3                            3                           
  caveat_summary     limited evidence, not        limited evidence, not       
                     decisive, needs review       decisive, needs review      


## 2 — All seven cases

Look for three patterns in the stored results: exact convergence, softer committee calibration, and cases where both paths keep strong caveats. The control case is the clean sanity check where both modes come out with a strong public verdict and light caveating.

In [25]:
results = [run_case(**c) for c in USE_CASES]

### How to read the table below

The comparison table keeps the public verdict, but the real separation between single and committee shows up in the continuous scores.

| Metric | What it means |
|---|---|
| `verdict` | Top-line public classification: `validated`, `contested`, `insufficient`, or `rejected` |
| `support` | Mean support score across the scored claim units |
| `pushback` | Mean contradiction score across the scored claim units |
| `confidence` | Mean sufficiency score across the scored claim units |
| `caveat_count` | Number of public caution flags attached to the result |
| `caveat_summary` | Compact explanation of the caution flags |

**How the stored results actually read:**
- **Forecast** — both modes stay cautious rather than claiming a strong validation.
- **AGI** — the committee can look less binary even when the headline verdict is similar, because the pushback and confidence scores move.
- **Control (diversification)** — this is the clean convergence case where both modes land on a stronger verdict with strong support and confidence.
- **Mixed-truth / High-stakes medical** — both modes keep caution visible, but the continuous scores show how sharply the committee pushes back.
- **Legal / AI risk** — both modes tend to preserve a severe or caution-heavy reading rather than overclaim certainty.

In [26]:
for r in results:
    print_case_summary(r)


──────────────────────────────────────────────────────────────────────────────────────
  Forecast: rate cut within two quarters
  metric             single                       committee                   
  ────────────────────────────────────────────────────────────────────────────────
  verdict            insufficient                 insufficient                
  support            0.0                          0.0                         
  pushback           0.0                          0.0                         
  confidence         0.0                          0.0                         
  caveat_count       3                            3                           
  caveat_summary     limited evidence, not        limited evidence, not       
                     decisive, needs review       decisive, needs review      

──────────────────────────────────────────────────────────────────────────────────────
  AGI: capability claim without operational definition
  metric     

#### _In this stored run, the committee is most informative when it softens a binary single-juror read rather than when it fully overturns it._

## What to look for

Both modes use the same model (`gpt-5.4-mini`), so differences come from committee architecture — four personality-injected personas vs one generic call. Each juror's system prompt includes its `Profile` fields (stance, evidence demand, safety bias), so any deltas here are due to reviewer composition rather than model class.

| Pattern | Where to expect it |
|---|---|
| Single and committee both stay cautious | Forecast under-evidence case |
| Committee gives a softer, less binary read | AGI claim and some medical overclaims |
| Single and committee converge cleanly | Control claim, plus the strongest legal / AI overclaims |

Read `verdict` as the headline, then use `support`, `pushback`, and `confidence` as the actual comparison signal. `caveat_summary` is there to explain the caution briefly without taking over the table. If you rerun the live cells, treat the text above as guidance for this saved run rather than a guarantee of identical future numbers.